# LangGraph G4 — Structured output and routing
CampusAI v2 answers in prose. A helpdesk system needs machine-readable decisions:

```json
{"category": "records", "priority": "high", "student_id": "S001"}
```

**Structured output** makes the model fill in a Pydantic schema instead of writing free text.
The object is validated before your code sees it. Two uses follow immediately: a **ticket** that a
queue can sort, and a **routing decision** that a conditional edge can act on.

```text
                      +-> faq (handbook search + model) --+
START -> triage ------+-> records (the G3 agent subgraph) -+--> END
                      +-> smalltalk (model only) ---------+
```

The records desk is the *whole agent graph from G3 used as a single node*: a **subgraph**.
This mix of a fixed workflow (triage, faq) with an agent inside it (records) is how most
production systems look: agents where judgement helps, workflow everywhere else.

### Step 1 — A ticket schema and `with_structured_output`

`Literal` fields constrain values; `Field(description=...)` tells the model what each field
means. `with_structured_output()` returns a model that produces the object directly.

In [ ]:
class Ticket(BaseModel):                                   # ours, on Pydantic's BaseModel
    """A classified helpdesk message."""
    category: Literal["faq", "records", "smalltalk"] = Field(description="faq for rules and general information; records for a specific student, course or campus service; smalltalk otherwise.")
    priority: Literal["low", "medium", "high"] = Field(description="high when an exam, a deadline or money is at stake.")
    student_id: str = Field(description="Student id if mentioned, otherwise 'unknown'.")

def structured(schema):                                    # ours: with_structured_output, portable across OpenRouter models
    return model.with_structured_output(schema, method="function_calling") if LIVE else model.with_structured_output(schema)   # LangChain

for text in ["Student S001 has 68% attendance and the exam is next week, is that a problem?", "What are the library hours?", "Hello there!"]:
    ticket = structured(Ticket).invoke([HumanMessage(text)])   # LangChain -> a validated Ticket object
    print(f"{type(ticket).__name__} {ticket.model_dump()}  <- {text[:45]}")   # Pydantic: object -> dict

### Step 2 — Triage node, desks, and the agent as a subgraph

The triage node stores the ticket fields in the state; the routing function reads the category.
A compiled graph can be added as a node: because both graphs share the `messages` key, the
subgraph reads the conversation and appends its answers.

In [ ]:
class DeskState(TypedDict, total=False):                   # ours: the workflow state (messages shared with the subgraph)
    messages: Annotated[list, add_messages]
    category: str
    priority: str

def triage(state: DeskState):                              # ours: node
    ticket = structured(Ticket).invoke([HumanMessage(text_of(state["messages"][-1]))])   # LangChain
    return {"category": ticket.category, "priority": ticket.priority}

def faq(state: DeskState):                                 # ours: fixed 2-step answer, no loop (upgraded to real retrieval in G7)
    excerpts = search_handbook.invoke({"query": text_of(state["messages"][-1])})   # LangChain: call the tool directly
    reply = model.invoke([SystemMessage("Answer only from the handbook excerpts below.\n\n" + excerpts), state["messages"][-1]])   # LangChain
    return {"messages": [reply]}

def smalltalk(state: DeskState):                           # ours
    return {"messages": [model.invoke([SystemMessage(CAMPUS_PERSONA)] + state["messages"])]}

records_agent = build_agent(READ_TOOLS)                    # ours: the G3 agent graph, compiled

def build_desks(faq_node=faq, records_node=records_agent, checkpointer=None):   # ours: reused and upgraded in later sections
    desk = StateGraph(DeskState)
    desk.add_node("triage", triage)
    desk.add_node("faq", faq_node)
    desk.add_node("records", records_node)                 # LangGraph: a compiled graph becomes a node (a subgraph)
    desk.add_node("smalltalk", smalltalk)
    desk.add_edge(START, "triage")
    desk.add_conditional_edges("triage", lambda state: state["category"], {"faq": "faq", "records": "records", "smalltalk": "smalltalk"})   # LangGraph
    for node in ("faq", "records", "smalltalk"):
        desk.add_edge(node, END)
    return desk.compile(checkpointer=checkpointer)

campusai_v3 = build_desks()
for q in ["Hello there!", "Can a failed course be retaken?", "How many credits does S002 have and is CS101 open?"]:
    out = campusai_v3.invoke({"messages": [HumanMessage(q)]})
    print(f"\n[{out['category']}/{out['priority']}] {q}")
    show_messages(out["messages"][1:])
print("\n" + campusai_v3.get_graph().draw_mermaid())

### Recap

- **Problem seen:** prose answers cannot be sorted or routed, and every message ran the full agent loop.
- **Layer added:** structured output with a Pydantic schema, a triage node, conditional routing, and the agent graph as a subgraph node.
- **Evidence:** three tickets came back as validated objects; three questions took three different paths.